# 02 — CSP Yield Landscape

**Was Du hier siehst:** Cross-Underlying-Vergleich der Cash-Secured-Put-Opportunities — best annualisierte Premium-Yield + IV-Median pro Tag pro Underlying, plus eine DTE×Buffer-Heatmap die zeigt wo die historischen Sweet-Spots sitzen.

**Datenquelle:**
- `mkt_options_snapshot` — wird täglich 23:05 vom `screener_csp` Side-Effect persistiert. Jeder Run schreibt *alle evaluierten* Strikes (nicht nur die Top-N), das gibt die Heatmap-Granularität.

**Annahmen:**
- Annualisierte Yield-Formel identisch zur Engine: `(bid / strike) × (365 / dte) × 100`.
- Buffer (=Out-of-the-money-Distanz) wird im Notebook berechnet: `(underlying_spot - strike) / underlying_spot × 100`.
- Nur Puts (`right = 'P'`).

**Wichtig:** Die `mkt_options_snapshot`-Tabelle akkumuliert ab dem ersten daemon-Run — Trends sind erst nach 5-10 Tagen aussagekräftig. Bis dahin zeigt das Notebook nur den Latest-Stand und meldet ehrlich "insufficient history".

In [ ]:
import sys, pathlib
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
import numpy as np
from datetime import date, timedelta

from modules.analysis._db import session, df
from modules.analysis._plot import setup, COLORS, hline, plt
setup()

In [ ]:
LOOKBACK_DAYS  = 30        # snapshot-history fenster
MIN_DTE_PLOT   = 15        # bound fuer DTE-Heatmap (matched typische CSP-Range)
MAX_DTE_PLOT   = 60
MIN_BUF_PLOT   = 0.0       # buffer in %
MAX_BUF_PLOT   = 25.0
MIN_HISTORY_FOR_TREND = 3  # weniger -> nur "latest snapshot" View, kein Trend-Plot

## 1. Snapshot-Daten laden + DB-Status-Check

Wenn die Tabelle leer ist oder noch keine 2 Tage Snapshots existieren, weiß Du das hier.

In [ ]:
since = date.today() - timedelta(days=LOOKBACK_DAYS)
with session() as con:
    overview = df(con, '''
        SELECT MIN(ts) AS first_ts, MAX(ts) AS last_ts,
               COUNT(DISTINCT ts) AS n_days,
               COUNT(DISTINCT ref_instrument_id) AS n_instruments,
               COUNT(*) AS n_rows
        FROM mkt_options_snapshot
        WHERE ts >= ?
    ''', [since])
overview

In [ ]:
with session() as con:
    snaps = df(con, '''
        SELECT s.ts, s.ref_instrument_id, s.expiration, s.strike, s."right",
               s.bid, s.ask, s.iv, s.volume, s.underlying_spot, s.dte,
               i.symbol
        FROM mkt_options_snapshot s
        LEFT JOIN ref_instruments i USING (ref_instrument_id)
        WHERE s.ts >= ?
          AND s."right" = 'P'
          AND s.bid IS NOT NULL AND s.bid > 0
          AND s.strike > 0 AND s.dte > 0
    ''', [since])
print(f'Rows: {len(snaps):,}   Days: {snaps["ts"].nunique() if not snaps.empty else 0}'
      f'   Underlyings: {snaps["ref_instrument_id"].nunique() if not snaps.empty else 0}')

if not snaps.empty:
    # Engine-konforme Yield + buffer berechnen.
    snaps['ann_yield_pct'] = (snaps['bid'] / snaps['strike']) * (365.0 / snaps['dte']) * 100.0
    snaps['buffer_pct']    = (snaps['underlying_spot'] - snaps['strike']) / snaps['underlying_spot'] * 100.0
snaps.head()

## 2. Best-Yield + IV-Median Trend pro Underlying

In [ ]:
if snaps.empty:
    print('Keine Snapshot-Daten — screener_csp muss laufen.')
elif snaps['ts'].nunique() < MIN_HISTORY_FOR_TREND:
    n_days = snaps['ts'].nunique()
    print(f'Nur {n_days} Tag(e) Snapshot-History — Trend braucht ≥ {MIN_HISTORY_FOR_TREND}. Zeige nur Latest-Snapshot.')
    latest_ts = snaps['ts'].max()
    latest = (snaps[snaps['ts'] == latest_ts]
             .groupby(['symbol', 'ref_instrument_id'], dropna=False)
             .agg(n_strikes=('strike', 'count'),
                  best_yield=('ann_yield_pct', 'max'),
                  median_yield=('ann_yield_pct', 'median'),
                  iv_median=('iv', 'median'))
             .reset_index()
             .sort_values('best_yield', ascending=False))
    print(f'Latest snapshot ({latest_ts}):')
    latest
else:
    daily = (snaps.groupby(['ref_instrument_id', 'symbol', 'ts'])
                  .agg(n_strikes=('strike', 'count'),
                       best_yield=('ann_yield_pct', 'max'),
                       median_yield=('ann_yield_pct', 'median'),
                       iv_median=('iv', 'median'))
                  .reset_index()
                  .sort_values(['ref_instrument_id', 'ts']))
    print(f'Daily aggregate rows: {len(daily)}')
    daily.head(10)

In [ ]:
if not snaps.empty and snaps['ts'].nunique() >= MIN_HISTORY_FOR_TREND:
    # Top-Underlyings nach mean(best_yield) auswaehlen — zu viele Linien sind unlesbar.
    top_underlyings = (daily.groupby('symbol')['best_yield'].mean()
                            .sort_values(ascending=False).head(6).index.tolist())
    plotted = daily[daily['symbol'].isin(top_underlyings)]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    for sym, g in plotted.groupby('symbol'):
        ax1.plot(g['ts'], g['best_yield'], marker='o', markersize=3, label=sym)
        ax2.plot(g['ts'], g['iv_median'], marker='o', markersize=3, label=sym)
    ax1.set_title(f'Best annualisierte Yield pro Tag (Top-{len(top_underlyings)} nach Mean)')
    ax1.set_ylabel('Best Yield (%/yr)')
    ax1.legend(loc='upper left', ncol=3, fontsize=8)
    ax2.set_title('IV-Median pro Tag')
    ax2.set_ylabel('IV (decimal)')
    ax2.set_xlabel('Datum')
    plt.tight_layout(); plt.show()

## 3. DTE × Buffer Sweet-Spot-Heatmap

Bucketed: DTE in 5-Tage-Steps, Buffer in 2.5-Pkt-Steps. Aggregat = mean(ann_yield_pct) über alle Underlyings × Snapshots. Sagt Dir: *welche Kombination liefert historisch die höchste Yield* (über Underlyings hinweg) — Underlying-spezifischen Filter machst Du in Cell 4.

In [ ]:
if not snaps.empty:
    h = snaps[(snaps['dte'].between(MIN_DTE_PLOT, MAX_DTE_PLOT)) &
              (snaps['buffer_pct'].between(MIN_BUF_PLOT, MAX_BUF_PLOT))].copy()
    if h.empty:
        print(f'Keine Snapshots in DTE [{MIN_DTE_PLOT}..{MAX_DTE_PLOT}] × Buffer [{MIN_BUF_PLOT}..{MAX_BUF_PLOT}].')
    else:
        h['dte_bucket']    = (h['dte'] // 5) * 5
        h['buffer_bucket'] = (h['buffer_pct'] // 2.5) * 2.5
        heat = (h.groupby(['dte_bucket', 'buffer_bucket'])['ann_yield_pct']
                  .agg(['mean', 'count'])
                  .reset_index())
        # Pivot fuer Heatmap.
        pivot_mean  = heat.pivot(index='buffer_bucket', columns='dte_bucket', values='mean').sort_index(ascending=False)
        pivot_count = heat.pivot(index='buffer_bucket', columns='dte_bucket', values='count').sort_index(ascending=False)
        print('Mean annualisierte Yield (%/yr):')
        pivot_mean

In [ ]:
if not snaps.empty and 'pivot_mean' in dir() and not h.empty:
    fig, ax = plt.subplots(figsize=(10, 5.5))
    im = ax.imshow(pivot_mean.values, aspect='auto', cmap='YlOrRd', origin='upper')
    ax.set_xticks(np.arange(len(pivot_mean.columns)))
    ax.set_xticklabels([f'{int(c)}d' for c in pivot_mean.columns])
    ax.set_yticks(np.arange(len(pivot_mean.index)))
    ax.set_yticklabels([f'{v:.1f}%' for v in pivot_mean.index])
    ax.set_xlabel('DTE (Bucket-Untergrenze)')
    ax.set_ylabel('Buffer (Bucket-Untergrenze, %)')
    ax.set_title('CSP Sweet-Spot: Mean annualisierte Yield (% p.a.)')
    cbar = plt.colorbar(im, ax=ax); cbar.set_label('Mean Yield (%/yr)')
    # Cell-Counts als Annotation.
    for i, by in enumerate(pivot_mean.index):
        for j, dt in enumerate(pivot_mean.columns):
            v = pivot_mean.iat[i, j]
            c = pivot_count.iat[i, j] if (i < pivot_count.shape[0] and j < pivot_count.shape[1]) else 0
            if pd.notna(v):
                ax.text(j, i, f'{v:.1f}\nn={int(c)}', ha='center', va='center',
                        fontsize=7, color='black' if v < pivot_mean.values[~np.isnan(pivot_mean.values)].mean() else 'white')
    plt.tight_layout(); plt.show()

## 4. Underlying-Drilldown

Setze `DRILL_SYMBOL` auf ein Ticker um die volle Strike × Expiration Time-Series für genau ein Underlying zu sehen. Default = das Underlying mit der höchsten mean(best_yield).

In [ ]:
if not snaps.empty:
    # Default-Pick.
    DRILL_SYMBOL = (snaps.groupby('symbol')['ann_yield_pct'].mean()
                         .sort_values(ascending=False).index[0])
    print(f'Drilldown auf: {DRILL_SYMBOL}')
    sub = snaps[snaps['symbol'] == DRILL_SYMBOL].copy()
    # Pro (expiration, strike) Time-Series an best-yield.
    pivot_yield = sub.pivot_table(index='ts', columns=['expiration', 'strike'],
                                   values='ann_yield_pct', aggfunc='max')
    fig, ax = plt.subplots(figsize=(11, 5.5))
    # Zeige Top-5 Strike-Expiration-Kombinationen nach mean(yield).
    if not pivot_yield.empty:
        top_cols = pivot_yield.mean().sort_values(ascending=False).head(5).index
        for col in top_cols:
            label = f'exp={col[0]} K={col[1]:.0f}'
            ax.plot(pivot_yield.index, pivot_yield[col], marker='o', markersize=3, label=label)
        ax.set_title(f'{DRILL_SYMBOL} — annualisierte Yield, Top-5 Strike-Exp-Kombinationen')
        ax.set_ylabel('Yield (%/yr)'); ax.legend(fontsize=8, loc='best')
        plt.tight_layout(); plt.show()
    else:
        print(f'Keine Daten für {DRILL_SYMBOL}.')

## 5. Latest-Snapshot Ranking (heutiger Tag)

Schneller Action-Take: was *heute* die besten Yields liefert.

In [ ]:
if not snaps.empty:
    latest_ts = snaps['ts'].max()
    latest = snaps[snaps['ts'] == latest_ts].copy()
    latest['spread_pct'] = np.where(latest['ask'] > 0,
                                     (latest['ask'] - latest['bid']) / latest['ask'] * 100.0, np.nan)
    cols = ['symbol', 'expiration', 'strike', 'dte', 'buffer_pct',
            'bid', 'ask', 'spread_pct', 'iv', 'ann_yield_pct']
    print(f'Top-20 (snapshot {latest_ts}) by ann_yield_pct:')
    latest.sort_values('ann_yield_pct', ascending=False)[cols].head(20)